In [1]:
#1.installing spark and setting up
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

import findspark
findspark.init()

print("Spark setup done!!")

Spark setup done!!


In [2]:
# STEP 2: Start Spark
#importing required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, when

# spinning up the spark session locally
spark = SparkSession.builder.appName("MySparkAssignment").getOrCreate()
print("Session started")

Session started


In [3]:
# STEP 3: Load Data
# reading our custom uploaded dataset.csv file into a dataFrame
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)

# viewing the column names and data types (schema)
print("Column Names and Data Types (Quick schema)")
df.printSchema()

# viewing the first few rows of the data
print("Displaying First Few Rows ")
df.show()

Column Names and Data Types (Quick schema)
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- transaction_date: string (nullable = true)

Displaying First Few Rows 
+---+------+----+---------+------+------+------------+-----------------+--------+----------------+
| id|  name| age| category|salary|region|subscription|            email|username|transaction_date|
+---+------+----+---------+------+------+------------+-----------------+--------+----------------+
|  1|  Alex|  23|     Tech| 50000| North|     Premium|  alex1@gmail.com|   alex1|      01-01-2024|
|  2|   Ben|  34|       HR| 45000|  East|       Basic|   ben2@gmail.com|    ben2|      02-01-2024|
|  3| Chris|NULL|Marketing| 38000|  

In [4]:
# STEP 4: Data Cleaning
# 1. remove duplicate rows from the dataframe
df_no_duplicates = df.dropDuplicates()

# 2. handle missing values (filling missing salaries with the average, dropping null ages)
mean_salary = df_no_duplicates.select(avg("salary")).first()[0]
df_filled = df_no_duplicates.na.fill({"salary": int(mean_salary)})
df_clean_nulls = df_filled.na.drop(subset=["age"])

# 3. check for incorrect or inconsistent data (fixing george's negative age to null)
df_cleaned = df_clean_nulls.withColumn("age", when(col("age") < 0, None).otherwise(col("age")))

print("Cleaned Data Output")
df_cleaned.show()

Cleaned Data Output
+---+------+----+---------+------+------+------------+-------------------+---------+----------------+
| id|  name| age| category|salary|region|subscription|              email| username|transaction_date|
+---+------+----+---------+------+------+------------+-------------------+---------+----------------+
| 22|Victor|  33|  Finance| 57000|  East|       Basic| victor22@gmail.com| victor22|      22-01-2024|
|116|  Zara|NULL|Marketing| 53000|  East|     Premium|  zara116@gmail.com|  zara116|      26-04-2024|
| 68|Hannah|  45|Marketing| 52000| North|     Premium| hannah68@gmail.com| hannah68|      09-03-2024|
|106| Priya|NULL|Marketing| 51000|  West|       Basic| priya106@gmail.com| priya106|      16-04-2024|
| 94| Diana|  42|     Tech| 70000| North|     Premium|  diana94@gmail.com|  diana94|      04-04-2024|
| 20|   Tom|  26|     Tech| 54000|  West|       Basic|    tom20@gmail.com|    tom20|      20-01-2024|
|112|Victor|  33|  Finance| 57000|  East|       Basic|victor11

In [5]:
# STEP 5: Filter Data
# applying simple conditions to filter by age, category, and region
# keeping records where age is 30 or older, and region is not South
df_filtered = df_cleaned.filter((col("age") >= 30) & (col("region") != "South"))

print("Filtered Data Output")
df_filtered.show()

Filtered Data Output
+---+------+---+---------+------+------+------------+-------------------+---------+----------------+
| id|  name|age| category|salary|region|subscription|              email| username|transaction_date|
+---+------+---+---------+------+------+------------+-------------------+---------+----------------+
| 22|Victor| 33|  Finance| 57000|  East|       Basic| victor22@gmail.com| victor22|      22-01-2024|
| 68|Hannah| 45|Marketing| 52000| North|     Premium| hannah68@gmail.com| hannah68|      09-03-2024|
| 94| Diana| 42|     Tech| 70000| North|     Premium|  diana94@gmail.com|  diana94|      04-04-2024|
|112|Victor| 33|  Finance| 57000|  East|       Basic|victor112@gmail.com|victor112|      22-04-2024|
| 62|   Ben| 34|       HR| 45000|  East|       Basic|    ben62@gmail.com|    ben62|      03-03-2024|
| 92|   Ben| 34|       HR| 45000|  East|       Basic|    ben92@gmail.com|    ben92|      02-04-2024|
| 52|Victor| 33|  Finance| 57000|  East|       Basic| victor52@gmail.c

In [6]:
# STEP 6: Transform Data
# changing data types (making sure age is explicitly cast as an integer)
df_transformed = df_filtered.withColumn("age", col("age").cast("integer"))

# optionally renaming columns if needed for clean layout
df_transformed = df_transformed.withColumnRenamed("name", "employee_name")

print("Transformed Data Schema Check")
df_transformed.printSchema()
df_transformed.show()

Transformed Data Schema Check
root
 |-- id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = false)
 |-- region: string (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- transaction_date: string (nullable = true)

+---+-------------+---+---------+------+------+------------+-------------------+---------+----------------+
| id|employee_name|age| category|salary|region|subscription|              email| username|transaction_date|
+---+-------------+---+---------+------+------+------------+-------------------+---------+----------------+
| 22|       Victor| 33|  Finance| 57000|  East|       Basic| victor22@gmail.com| victor22|      22-01-2024|
| 68|       Hannah| 45|Marketing| 52000| North|     Premium| hannah68@gmail.com| hannah68|      09-03-2024|
| 94|        Diana| 4

In [7]:
# STEP 7: Aggregation
from pyspark.sql.functions import avg, min, max

# performing basic calculations across total rows
print(f"Total rows remaining: {df_transformed.count()}")

# finding average, minimum, and maximum values of the salary column
df_transformed.select(
    avg("salary").alias("average_salary"), min("salary").alias("minimum_salary"),max("salary").alias("maximum_salary")
).show()

Total rows remaining: 52
+-----------------+--------------+--------------+
|   average_salary|minimum_salary|maximum_salary|
+-----------------+--------------+--------------+
|59168.07692307692|         45000|         70000|
+-----------------+--------------+--------------+



In [8]:
# STEP 8: Group Data

from pyspark.sql.functions import count, avg, sum
# using groupBy() to group the filtered data by category
print("--- Grouped Data Summary ---")
df_transformed.groupBy("category").agg(
    count("id").alias("total_count"), avg("salary").alias("average_salary"), sum("salary").alias("total_salary_pool")
).show()

--- Grouped Data Summary ---
+---------+-----------+--------------+-----------------+
| category|total_count|average_salary|total_salary_pool|
+---------+-----------+--------------+-----------------+
|       HR|          8|       46000.0|           368000|
|  Finance|         12|       61000.0|           732000|
|     Tech|         20|       65237.0|          1304740|
|Marketing|         12|       56000.0|           672000|
+---------+-----------+--------------+-----------------+



In [9]:
# STEP 10: Build a Simple Pipeline
# combining all the steps into one final continuous flow and saving it
from pyspark.sql.functions import col, sum

raw_df = spark.read.csv("dataset.csv", header=True, inferSchema=True) # Load

# Clean & Transform steps combined
cleaned_pipeline_df = raw_df.dropDuplicates() \
    .na.fill({"salary": 0}) \
    .withColumn("age", col("age").cast("integer"))

# Filter step
filtered_pipeline_df = cleaned_pipeline_df.filter(col("age") >= 30)

# Aggregate step
final_aggregated_df = filtered_pipeline_df.groupBy("region").agg(sum("salary").alias("total_payroll"))

# Showing the pipeline output
final_aggregated_df.show()

# saving the final aggregated results folder
final_aggregated_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("output_result")
print("Pipeline complete. Output saved.")

+------+-------------+
|region|total_payroll|
+------+-------------+
| South|       456000|
|  East|       596000|
|  West|       784000|
| North|      1480000|
+------+-------------+

Pipeline complete. Output saved.


In [10]:
# checking final schema status to confirm validation steps
df_transformed.printSchema()
print("Verification check finished.")

root
 |-- id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = false)
 |-- region: string (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- transaction_date: string (nullable = true)

Verification check finished.
